# AG News Text Classification with Qwen2-1.5B + LoRA/QLoRA

This notebook fine-tunes **Qwen2-1.5B** for AG News **text classification** using **LoRA** and **QLoRA**.



In [1]:
!pip -q install -U transformers datasets peft accelerate bitsandbytes evaluate torchao>=0.16.0

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

## Load AG News

In [ ]:
dataset = load_dataset('ag_news')
dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

In [ ]:
label_list = dataset['train'].features['label'].names
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}
label_list

['World', 'Sports', 'Business', 'Sci/Tech']

## Tokenization

In [ ]:
model_checkpoint = 'Qwen/Qwen2-1.5B'
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, use_fast=True)

# Fix: ensure pad token exists for batching
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def preprocess(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=256
    )

tokenized = dataset.map(preprocess, batched=True, remove_columns=['text'])
tokenized

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 7600
    })
})

## Metrics

In [8]:
import evaluate
accuracy = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return accuracy.compute(predictions=preds, references=labels)

# 1) LoRA fine-tuning

In [11]:
from transformers import TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model

model_lora = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    label2id=label2id,
    id2label=id2label,
    torch_dtype=torch.float16
)

model_lora.config.pad_token_id = tokenizer.pad_token_id

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias='none',
    task_type='SEQ_CLS',
    target_modules=['q_proj', 'v_proj']
)

model_lora = get_peft_model(model_lora, lora_config)
model_lora.print_trainable_parameters()

args_lora = TrainingArguments(
    output_dir='outputs/lora',
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    max_steps=1000,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="no",
    learning_rate=2e-4,
    fp16=False,
    remove_unused_columns=False,
)

trainer_lora = Trainer(
    model=model_lora,
    args=args_lora,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['test'],
    compute_metrics=compute_metrics,
)

trainer_lora.train()
trainer_lora.evaluate()

model_lora.save_pretrained('outputs/lora')
tokenizer.save_pretrained('outputs/lora')

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2-1.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 1,095,680 || all params: 1,544,816,128 || trainable%: 0.0709


Step,Training Loss,Validation Loss


Step,Training Loss,Validation Loss,Accuracy
200,No log,nan,0.250000
400,No log,nan,0.250000
600,0.000000,nan,0.250000
800,0.000000,nan,0.250000
1000,0.000000,nan,0.250000


Training Loss,Validation Loss,Step,Accuracy
0.000000,nan,1000,0.250000


('outputs/lora/tokenizer_config.json',
 'outputs/lora/chat_template.jinja',
 'outputs/lora/tokenizer.json')

# 2) QLoRA fine-tuning (4-bit)
Loads the base model in 4-bit and applies LoRA adapters.

In [12]:
from transformers import BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

model_qlora = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    label2id=label2id,
    id2label=id2label,
    quantization_config=bnb_config,
    device_map='auto'
)

model_qlora.config.pad_token_id = tokenizer.pad_token_id

model_qlora = prepare_model_for_kbit_training(model_qlora)
model_qlora = get_peft_model(model_qlora, lora_config)
model_qlora.print_trainable_parameters()

args_qlora = TrainingArguments(
    output_dir='outputs/lora',
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    max_steps=1000,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="no",
    learning_rate=2e-4,
    fp16=False,
    remove_unused_columns=False,
)

trainer_qlora = Trainer(
    model=model_qlora,
    args=args_qlora,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['test'],
    compute_metrics=compute_metrics,
)

trainer_qlora.train()
trainer_qlora.evaluate()

model_qlora.save_pretrained('outputs/qlora')
tokenizer.save_pretrained('outputs/qlora')

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2-1.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 1,095,680 || all params: 1,544,816,128 || trainable%: 0.0709


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss


KeyboardInterrupt: 